# Bibliotecas

In [18]:
import mysql.connector

import undetected_chromedriver as uc

from selenium import webdriver  
from selenium.webdriver.chrome.service import Service 
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

from webdriver_manager.chrome import ChromeDriverManager

from collections import Counter

import spacy

from fuzzywuzzy import process

# Banco De Dados

## Conectando No Banco De Dados

In [19]:
conexao = mysql.connector.connect(
    host = 'localhost',
    user = 'root',
    password = 'BancoDeDado123',
    database = 'projetoUnimar'
)

## Criando o Cursor Para Manipular o Banco De Dados

In [20]:
cursor = conexao.cursor()

# Scraping Do Reclame Aqui!

## Inicializando selenium e webdriver manager

In [21]:
service = Service(ChromeDriverManager().install())  

options = webdriver.ChromeOptions()
#options.add_argument("--headless")

driver = uc.Chrome(options=options)

nlp_ner = spacy.load("Treinar IA/model-last") 

## Coletando os dados do reclame aqui

In [ ]:
produtos_conhecidos = [
    "lâmpada", "controle universal", "notebook", "tablet", "luminária", "smart robô aspirador wi-fi laser"
]

def normalizar_produto(produto):
    produto = produto.lower()
    doc = nlp_ner(produto)
    
    lematizado = " ".join([token.lemma_ for token in doc])
    
    if not lematizado.strip():
        lematizado = produto

    if lematizado.endswith('s'):
        lematizado = lematizado.rstrip('s')
    
    produto_correspondente, similaridade = process.extractOne(lematizado, produtos_conhecidos, score_cutoff=90)
    
    if produto_correspondente:
        return produto_correspondente
    
    return lematizado

empresa = input("Digite a empresa que deseje fazer scraping: ")

quantidade_pagina = int(input("Digite a quantidade de páginas que deseja pegar: "))

produtos_reclamados = []

dicionarioEmpresas = {
    "positivo" : "positivo-informatica",
    "habibs" : "habibs"
}

for pagina in range(quantidade_pagina):
    url = f"https://www.reclameaqui.com.br/empresa/{dicionarioEmpresas.get(empresa)}/lista-reclamacoes/?pagina={pagina + 1}"
    driver.get(url)
    
    WebDriverWait(driver, 10).until(lambda d: len(d.find_elements(By.CLASS_NAME, 'sc-1pe7b5t-1')) > 0)
    
    reclamacoes = driver.find_elements(By.CLASS_NAME, 'sc-1pe7b5t-1')
    
    for index in range(len(reclamacoes)):
        try:
            reclamacoes = driver.find_elements(By.CLASS_NAME, 'sc-1pe7b5t-1')
            reclamacao = reclamacoes[index]

            driver.execute_script("arguments[0].click();", reclamacao)
            
            WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CLASS_NAME, 'sc-lzlu7c-3')))
            
            titulo = driver.find_element(By.CLASS_NAME, 'sc-lzlu7c-3').text
            reclamacao = driver.find_element(By.CLASS_NAME, 'sc-lzlu7c-17').text.replace('\n', ' ')
            local_reclamacao = driver.find_elements(By.CLASS_NAME, 'sc-lzlu7c-6')[0].text
            data_reclamacao = driver.find_elements(By.CLASS_NAME, 'sc-lzlu7c-6')[1].text
            status_reclamacao = driver.find_element(By.CLASS_NAME, 'sc-lzlu7c-18').text
            
            produto_identificado = None
            motivo_identificado = None

            doc = nlp_ner(reclamacao)
            for ent in doc.ents:
                if ent.label_ == "PRODUTO":
                    produto_identificado = ent.text
                    break

            for ent in doc.ents:
                if ent.label_ == "MOTIVO":
                    motivo_identificado = ent.text
                    break
            
            if produto_identificado:
                produto_normalizado = normalizar_produto(produto_identificado)
                produtos_reclamados.append(produto_normalizado)
            
            comando = f'INSERT INTO positivo (titulo_reclamacao, reclamacao, local_reclamacao, data_reclamacao, status_reclamacao, produto_reclamado, motivo_reclamado) VALUES ("{titulo}", "{reclamacao}", "{local_reclamacao}", "{data_reclamacao}", "{status_reclamacao}", "{produto_normalizado}", "{motivo_identificado}")'
            cursor.execute(comando)
            conexao.commit()
            
            driver.get(url)
            
            WebDriverWait(driver, 10).until(lambda d: len(d.find_elements(By.CLASS_NAME, 'sc-1pe7b5t-1')) > 0)
        
        except:
            driver.get(url)

contador_produtos = Counter(produtos_reclamados)

ranking_produtos = contador_produtos.most_common()

for produto, frequencia in ranking_produtos:
    print(f"{produto}: {frequencia} reclamações")

cursor.close()
conexao.close()
driver.quit()

NoSuchWindowException: Message: no such window: target window already closed
from unknown error: web view not found
  (Session info: chrome=131.0.6778.70)
Stacktrace:
	GetHandleVerifier [0x00EC5093+25075]
	(No symbol) [0x00E4E124]
	(No symbol) [0x00D2BE63]
	(No symbol) [0x00D0D92B]
	(No symbol) [0x00D97F7F]
	(No symbol) [0x00DAADB9]
	(No symbol) [0x00D91C16]
	(No symbol) [0x00D63F3C]
	(No symbol) [0x00D64ECD]
	GetHandleVerifier [0x011B2523+3094147]
	GetHandleVerifier [0x011C5754+3172532]
	GetHandleVerifier [0x011BDF32+3141778]
	GetHandleVerifier [0x00F62100+668256]
	(No symbol) [0x00E56C4D]
	(No symbol) [0x00E53DF8]
	(No symbol) [0x00E53F95]
	(No symbol) [0x00E46C80]
	BaseThreadInitThunk [0x76AB5D49+25]
	RtlInitializeExceptionChain [0x77A8CEBB+107]
	RtlGetAppContainerNamedObjectPath [0x77A8CE41+561]
